# ML: Customer Churn Prediction with Spark ML & MLflow

Predicts customers at risk of churning using a distributed Spark ML
`GBTClassifier` tracked with Fabric-native MLflow.

## Data Flow
```
Silver (fact_receipts, dim_customers) --> Feature Engineering --> GBTClassifier (distributed)
                                                                      |
                                              +-----------------------+
                                              v                       v
                                churn_predictions (Gold)  MLflow Experiment
```

## Model Details
- **Algorithm:** Spark ML `GBTClassifier` (distributed, Fabric-native)
- **Target:** Binary churn (no purchase in configurable window)
- **Features:** Behavioral (purchase patterns) + customer geography
- **Tracking:** MLflow experiment with parameters, metrics, and logged model
- **Target metrics:** AUC-ROC > 0.75, Precision > 0.6

## Usage
Schedule this notebook on a configurable cadence (weekly is a common default)
to refresh churn predictions.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.functions import vector_to_array
from datetime import datetime, timedelta, timezone
import mlflow
import os

In [ ]:
# =============================================================================
# PARAMETERS
# =============================================================================

def get_env(var_name, default=None):
    return os.environ.get(var_name, default)

LAKEHOUSE_NAME = get_env("LAKEHOUSE_NAME", default="retail_lakehouse")
SILVER_DB = get_env("SILVER_DB", default="silver")
GOLD_DB = get_env("GOLD_DB", default="gold")
EXPERIMENT_NAME = get_env("MLFLOW_EXPERIMENT", default="churn_prediction")
RECEIPTS_TABLE = get_env("RECEIPTS_TABLE", default="fact_receipts")
CUSTOMERS_TABLE = get_env("CUSTOMERS_TABLE", default="dim_customers")
CHURN_PREDICTIONS_TABLE = get_env("CHURN_PREDICTIONS_TABLE", default="churn_predictions")
CHURN_PREDICTIONS_TABLE_NAME = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{CHURN_PREDICTIONS_TABLE}"
SCHEDULE_CADENCE = get_env("SCHEDULE_CADENCE", default="weekly")

# Churn definition (days without purchase) - reduced for synthetic data diversity
CHURN_WINDOW_DAYS = int(get_env("CHURN_WINDOW_DAYS", default="30"))

# Feature engineering window (days of historical data to analyze)
FEATURE_WINDOW_DAYS = int(get_env("FEATURE_WINDOW_DAYS", default="730"))

# Training window (exclude recent data for label stability)
LABEL_OFFSET_DAYS = int(get_env("LABEL_OFFSET_DAYS", default="7"))

# GBT hyperparameters
MAX_ITER = int(get_env("MAX_ITER", default="80"))
MAX_DEPTH = int(get_env("MAX_DEPTH", default="5"))
STEP_SIZE = float(get_env("STEP_SIZE", default="0.1"))
SUBSAMPLING_RATE = float(get_env("SUBSAMPLING_RATE", default="1.0"))



print(f"Configuration: SILVER_DB={SILVER_DB}, GOLD_DB={GOLD_DB}")
print(f"Source tables: {LAKEHOUSE_NAME}.{SILVER_DB}.{RECEIPTS_TABLE}, {LAKEHOUSE_NAME}.{SILVER_DB}.{CUSTOMERS_TABLE}")
print(f"Output table: {CHURN_PREDICTIONS_TABLE_NAME}")
print(f"MLflow experiment: {EXPERIMENT_NAME}")
print(f"Churn window: {CHURN_WINDOW_DAYS} days")
print(f"Feature window: {FEATURE_WINDOW_DAYS} days")
print(f"Schedule cadence: {SCHEDULE_CADENCE}")
print(f"GBTClassifier: maxIter={MAX_ITER}, maxDepth={MAX_DEPTH}, stepSize={STEP_SIZE}, subsamplingRate={SUBSAMPLING_RATE}")

In [ ]:
# =============================================================================
# HELPER FUNCTIONS & MLFLOW SETUP
# =============================================================================

def ensure_database(name):
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {LAKEHOUSE_NAME}.{name}")

def read_silver(table_name):
    return spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")

def silver_exists(table_name):
    try:
        spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")
        return True
    except AnalysisException:
        return False

def resolve_table_column(df, table_name, *candidates):
    available = {column.lower(): column for column in df.columns}
    for candidate in candidates:
        resolved = available.get(candidate.lower())
        if resolved is not None:
            return resolved
    raise RuntimeError(
        f"Unable to resolve any of {candidates} in {table_name}. Available columns: {df.columns}"
    )

ensure_database(GOLD_DB)

# Fabric auto-configures the MLflow tracking URI
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow experiment '{EXPERIMENT_NAME}' ready")

## Data Validation & Preparation

In [ ]:
required_tables = [RECEIPTS_TABLE, CUSTOMERS_TABLE]
for table in required_tables:
    if not silver_exists(table):
        raise RuntimeError(f"Required table {LAKEHOUSE_NAME}.{SILVER_DB}.{table} not found!")
    print(f"  {table}: OK")

print("\nAll required tables exist.")

In [ ]:
print("Determining analysis dates...")

latest_date = (
    read_silver(RECEIPTS_TABLE)
    .select(F.max(F.to_date("event_ts")).alias("latest_date"))
    .first()[0]
)

if latest_date is None:
    raise RuntimeError(f"No transaction data found in {RECEIPTS_TABLE}!")

snapshot_date = latest_date - timedelta(days=LABEL_OFFSET_DAYS)
feature_start_date = snapshot_date - timedelta(days=FEATURE_WINDOW_DAYS)
churn_cutoff_date = snapshot_date - timedelta(days=CHURN_WINDOW_DAYS)

print(f"  Latest transaction date: {latest_date}")
print(f"  Snapshot date (for predictions): {snapshot_date}")
print(f"  Feature window: {feature_start_date} to {snapshot_date}")
print(f"  Churn cutoff (last purchase before): {churn_cutoff_date}")

## Feature Engineering

Build behavioral features from purchase history and customer dimension
features from the available customer table columns. All operations use
Spark — fully distributed.

In [ ]:
print("Loading transaction data...")
snapshot_date_lit = F.lit(snapshot_date).cast("date")
feature_start_date_lit = F.lit(feature_start_date).cast("date")
churn_cutoff_date_lit = F.lit(churn_cutoff_date).cast("date")

receipts_df = (
    read_silver(RECEIPTS_TABLE)
    .select(
        "customer_id",
        "store_id",
        F.to_date("event_ts").alias("event_date"),
        "total_amount",
        "payment_method",
    )
    .cache()
)
_ = receipts_df.count()

print("Loading customer dimension...")
customer_source_df = read_silver(CUSTOMERS_TABLE)
customer_id_col = resolve_table_column(
    customer_source_df,
    CUSTOMERS_TABLE,
    "customer_id",
    "id",
)
geography_id_col = resolve_table_column(
    customer_source_df,
    CUSTOMERS_TABLE,
    "geography_id",
    "geographyid",
)
customers_df = customer_source_df.select(
    F.col(customer_id_col).alias("customer_id"),
    F.col(geography_id_col).alias("geography_id"),
)

# Behavioral features: purchase patterns in feature window
print("\nEngineering behavioral features...")
behavioral_features = (
    receipts_df
    .filter(
        (F.col("event_date") >= feature_start_date_lit)
        & (F.col("event_date") <= snapshot_date_lit)
    )
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("purchase_count"),
        F.countDistinct("store_id").alias("unique_stores"),
        F.sum("total_amount").alias("total_spend"),
        F.avg("total_amount").alias("avg_basket_value"),
        F.stddev("total_amount").alias("basket_std"),
        F.max("total_amount").alias("max_basket"),
        F.min("total_amount").alias("min_basket"),
        F.max("event_date").alias("last_purchase_date"),
        F.countDistinct("payment_method").alias("payment_methods_used"),
    )
    .withColumn(
        "days_since_last_purchase",
        F.datediff(snapshot_date_lit, F.col("last_purchase_date")),
    )
    .withColumn(
        "purchase_frequency",
        F.col("purchase_count") / F.lit(FEATURE_WINDOW_DAYS),
    )
    .withColumn(
        "basket_consistency",
        F.when(F.col("basket_std").isNull(), 0.0)
        .otherwise(F.col("basket_std") / F.col("avg_basket_value")),
    )
    .drop("last_purchase_date")
)

print(f"  Behavioral features created for {behavioral_features.count()} customers")

# Customer dimension features
print("\nEngineering customer dimension features...")
demographic_features = customers_df.select("customer_id", "geography_id")

print(
    f"  Customer dimension features created for {demographic_features.count()} customers"
)

# Combine all features
print("\nCombining features...")
features_df = (
    demographic_features
    .join(behavioral_features, on="customer_id", how="left")
    .fillna({
        "purchase_count": 0,
        "unique_stores": 0,
        "total_spend": 0.0,
        "avg_basket_value": 0.0,
        "basket_std": 0.0,
        "max_basket": 0.0,
        "min_basket": 0.0,
        "days_since_last_purchase": FEATURE_WINDOW_DAYS + 1,
        "payment_methods_used": 0,
        "purchase_frequency": 0.0,
        "basket_consistency": 0.0,
    })
)

print(f"  Combined features for {features_df.count()} customers")

In [ ]:
# =============================================================================
# CREATE TARGET VARIABLE (CHURN LABEL)
# =============================================================================

print("Creating churn labels...")

churned_customers = (
    receipts_df
    .filter(
        (F.col("event_date") > churn_cutoff_date_lit)
        & (F.col("event_date") <= snapshot_date_lit)
    )
    .select("customer_id")
    .distinct()
)

dataset_df = (
    features_df
    .join(
        churned_customers.withColumn("is_active", F.lit(1)),
        on="customer_id",
        how="left",
    )
    .withColumn("is_churned", F.when(F.col("is_active").isNull(), 1).otherwise(0))
    .drop("is_active")
)

# Check class balance
churn_stats = dataset_df.groupBy("is_churned").count().collect()
class_counts = {int(row["is_churned"]): int(row["count"]) for row in churn_stats}
total_customers = sum(class_counts.values())
active_count = class_counts.get(0, 0)
churned_count = class_counts.get(1, 0)

if total_customers == 0:
    raise RuntimeError("No customers available for churn training after feature assembly.")

churn_rate = churned_count / total_customers

print(f"  Total customers: {total_customers}")
print(f"  Churned: {churned_count} ({churn_rate:.1%})")
print(f"  Active: {active_count} ({1 - churn_rate:.1%})")

if active_count == 0 or churned_count == 0:
    raise RuntimeError(
        "Churn training dataset contains only one class for the current analysis windows. "
        "Increase history or adjust FEATURE_WINDOW_DAYS, CHURN_WINDOW_DAYS, or LABEL_OFFSET_DAYS."
    )

if active_count < 2 or churned_count < 2:
    raise RuntimeError(
        "Churn training dataset needs at least 2 customers in each class to create train/test splits. "
        "Increase history or adjust FEATURE_WINDOW_DAYS, CHURN_WINDOW_DAYS, or LABEL_OFFSET_DAYS."
    )


## Model Training & Evaluation

Assemble features with `VectorAssembler`, train a distributed Spark ML
`GBTClassifier`, and evaluate with Spark evaluators — all tracked
in MLflow.

In [ ]:
# Define feature columns
feature_cols = [
    "geography_id",
    "purchase_count",
    "unique_stores",
    "total_spend",
    "avg_basket_value",
    "basket_std",
    "max_basket",
    "min_basket",
    "days_since_last_purchase",
    "payment_methods_used",
    "purchase_frequency",
    "basket_consistency",
]

print(f"Features ({len(feature_cols)}): {', '.join(feature_cols)}")

# Cast all feature columns to double for Spark ML
for col_name in feature_cols:
    dataset_df = dataset_df.withColumn(col_name, F.col(col_name).cast("double"))
dataset_df = dataset_df.na.fill(0.0, subset=feature_cols)
dataset_df = dataset_df.withColumn("is_churned", F.col("is_churned").cast("double"))

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep",
)
df_assembled = assembler.transform(dataset_df)

# Stratified train/test split to preserve both classes in each split
split_window = Window.partitionBy("is_churned").orderBy(F.rand(seed=42))
class_window = Window.partitionBy("is_churned")

df_ranked = (
    df_assembled
    .withColumn("class_row_num", F.row_number().over(split_window))
    .withColumn("class_count", F.count("*").over(class_window))
    .withColumn(
        "test_row_count",
        F.greatest(
            F.lit(1),
            F.floor(F.col("class_count") * F.lit(0.2)).cast("int"),
        ),
    )
)

df_test = (
    df_ranked
    .filter(F.col("class_row_num") <= F.col("test_row_count"))
    .drop("class_row_num", "class_count", "test_row_count")
)
df_train = (
    df_ranked
    .filter(F.col("class_row_num") > F.col("test_row_count"))
    .drop("class_row_num", "class_count", "test_row_count")
)

train_count = df_train.count()
test_count = df_test.count()

if train_count == 0 or test_count == 0:
    raise RuntimeError(
        "Unable to create non-empty stratified train/test splits. "
        "Increase history or adjust FEATURE_WINDOW_DAYS, CHURN_WINDOW_DAYS, or LABEL_OFFSET_DAYS."
    )

print(f"Train rows: {train_count}")
print(f"Test rows: {test_count}")

# Configure Spark ML GBT classifier
print("\nTraining GBTClassifier model (distributed across Spark executors)...")

gbt = GBTClassifier(
    featuresCol="features",
    labelCol="is_churned",
    maxIter=MAX_ITER,
    maxDepth=MAX_DEPTH,
    stepSize=STEP_SIZE,
    subsamplingRate=SUBSAMPLING_RATE,
    seed=42,
)

with mlflow.start_run(
    run_name=f"gbt_churn_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M')}"
) as run:
    mlflow.log_params({
        "algorithm": "gbt_classifier",
        "churn_window_days": CHURN_WINDOW_DAYS,
        "feature_window_days": FEATURE_WINDOW_DAYS,
        "max_iter": MAX_ITER,
        "max_depth": MAX_DEPTH,
        "step_size": STEP_SIZE,
        "subsampling_rate": SUBSAMPLING_RATE,
        "feature_count": len(feature_cols),
        "train_rows": train_count,
        "test_rows": test_count,
    })

    model = gbt.fit(df_train.select("features", "is_churned"))

    # ── Evaluate on held-out test set ───────────────────────────
    df_preds = model.transform(df_test).withColumnRenamed("prediction", "churn_prediction")

    # Extract churn probability from probability vector (index 1 = positive class)
    df_preds = df_preds.withColumn(
        "churn_probability", vector_to_array("probability")[1]
    )

    # AUC-ROC
    binary_evaluator = BinaryClassificationEvaluator(
        labelCol="is_churned",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC",
    )
    auc_roc = binary_evaluator.evaluate(df_preds)

    # Precision, Recall, F1
    mc_evaluator = MulticlassClassificationEvaluator(
        labelCol="is_churned",
        predictionCol="churn_prediction",
    )
    precision = mc_evaluator.evaluate(df_preds, {mc_evaluator.metricName: "weightedPrecision"})
    recall = mc_evaluator.evaluate(df_preds, {mc_evaluator.metricName: "weightedRecall"})
    f1 = mc_evaluator.evaluate(df_preds, {mc_evaluator.metricName: "f1"})

    mlflow.log_metrics({
        "auc_roc": round(auc_roc, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "churn_rate": round(churn_rate, 4),
    })

    # Log model
    mlflow.spark.log_model(model, "gbt_churn_model")

    print(f"\nMLflow run: {run.info.run_id}")
    print(f"\nTest-set evaluation:")
    print(f"  AUC-ROC:   {auc_roc:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")

    print(f"\nAcceptance Criteria:")
    print(f"  AUC-ROC > 0.75: {'PASS' if auc_roc > 0.75 else 'FAIL'} ({auc_roc:.4f})")
    print(f"  Precision > 0.6: {'PASS' if precision > 0.6 else 'FAIL'} ({precision:.4f})")

    if auc_roc > 0.75 and precision > 0.6:
        print("\n  All acceptance criteria met")
    else:
        print("\n  Below target — consider tuning hyperparameters or adding features")

## Save Predictions to Gold Layer

Score the full dataset and write predictions to Delta.

In [ ]:
print("Scoring full dataset...")

# Score every customer using the trained model
df_scored = model.transform(df_assembled).withColumnRenamed("prediction", "churn_prediction")
df_scored = df_scored.withColumn(
    "churn_probability", vector_to_array("probability")[1]
)

# Build risk categories using Spark
df_scored = df_scored.withColumn(
    "risk_category",
    F.when(F.col("churn_probability") < 0.2, "Very Low")
    .when(F.col("churn_probability") < 0.4, "Low")
    .when(F.col("churn_probability") < 0.6, "Medium")
    .when(F.col("churn_probability") < 0.8, "High")
    .otherwise("Very High"),
)

# Select final output columns
prediction_date = datetime.now(timezone.utc)
model_version = f"gbt_{run.info.run_id[:8]}"

df_final = (
    df_scored
    .select(
        F.col("customer_id").cast("long"),
        F.col("churn_probability").cast("double"),
        F.col("churn_prediction").cast("int"),
        F.col("risk_category").cast("string"),
        F.col("is_churned").alias("is_churned_actual").cast("int"),
        F.lit(prediction_date).alias("prediction_date").cast("timestamp"),
        F.lit(model_version).alias("model_version").cast("string"),
        F.lit(CHURN_WINDOW_DAYS).alias("churn_window_days").cast("int"),
    )
)

# Save to Gold layer
table_name = CHURN_PREDICTIONS_TABLE_NAME
df_final.write.format("delta").mode("overwrite").saveAsTable(table_name)

saved_count = spark.table(table_name).count()
print(f"Saved {saved_count} prediction records to {table_name}")

# Risk category distribution
print("\nRisk Category Distribution:")
risk_dist = (
    spark.table(table_name)
    .groupBy("risk_category")
    .agg(F.count("*").alias("count"))
    .orderBy("risk_category")
    .collect()
)
for row in risk_dist:
    pct = row["count"] / saved_count * 100
    print(f"  {row['risk_category']:<12} {row['count']:>6} ({pct:>5.1f}%)")

In [ ]:
print("=" * 60)
print("CHURN PREDICTION COMPLETE")
print("=" * 60)

gold_tables = spark.sql(f"SHOW TABLES IN {LAKEHOUSE_NAME}.{GOLD_DB}").collect()
print(f"\nGold layer ({GOLD_DB}): {len(gold_tables)} tables")
print(f"Prediction table: {CHURN_PREDICTIONS_TABLE_NAME}")
print(f"Churn window: {CHURN_WINDOW_DAYS} days")
print(f"\nMLflow experiment: {EXPERIMENT_NAME}")
print(f"Model logged: gbt_churn_model")
print("View results in Fabric > Experiments")
print(f"\nSchedule this notebook to run on a {SCHEDULE_CADENCE} cadence for fresh predictions.")